# LeakAudit - 07 - Foundation models under protocol shift (E7)

**Do modern audio representation models escape the collapse, or memorise more?**

- A reviewer's first objection is that only shallow models were tested. This notebook answers it with frozen embeddings from pre-trained audio models.
- Extractors are tried in order of download reliability; the 2023 mel pipeline is always included as the in-house baseline.
- Prediction: stronger encoders raise the naive score but collapse further under logger-independent evaluation - representational power spent on fingerprints.


In [2]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import src as la
from src import config as cfg, io_utils, features, evaluation
cfg.apply_mpl_style()
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split, GroupShuffleSplit

M2 = io_utils.scan_d2()
D2W = M2[(M2.ext == ".wav") & (M2.label.isin(["leak", "noleak"]))].reset_index(drop=True)
print(f"D2 wav: {len(D2W)}   loggers: {D2W.logger.nunique()}")

D2 wav: 140   loggers: 50


## Build the extractor set

The in-house 2023 mel pipeline always works. Foundation-model extractors
(torchaudio bundles first, then Hugging Face AST/CLAP) are added when
available; failures are caught so the notebook still runs offline.

In [3]:
EXTRACTORS = {}

# (0) In-house 2023 mel-spectrogram pipeline -- always available.
def embed_mel2023(x, sr):
    xb = features.bandpass(x, sr)
    xw = features.pick_window(xb, sr, sec=1.0, mode="minstd")
    return features.summary_features(xw, sr)
EXTRACTORS["Mel2023"] = embed_mel2023

USE_FOUNDATION = True
if USE_FOUNDATION:
    try:
        import torch, librosa
        DEV = "cuda" if torch.cuda.is_available() else "cpu"

        # (1) torchaudio self-supervised bundles (robust download path).
        for bundle_name in ["WAV2VEC2_BASE", "HUBERT_BASE"]:
            try:
                import torchaudio
                bundle = getattr(torchaudio.pipelines, bundle_name)
                model = bundle.get_model().to(DEV).eval()
                sr_req = bundle.sample_rate
                def make_fn(m, srq):
                    def f(x, sr):
                        xr = librosa.resample(x, orig_sr=sr, target_sr=srq) if sr != srq else x
                        with torch.no_grad():
                            t = torch.tensor(xr, dtype=torch.float32, device=DEV)[None]
                            feats, _ = m.extract_features(t)
                            h = feats[-1]
                            return torch.cat([h.mean(1), h.std(1)], -1).cpu().numpy().ravel()
                    return f
                EXTRACTORS[bundle_name] = make_fn(model, sr_req)
                print(f"  {bundle_name} ready")
            except Exception as e:
                print(f"  {bundle_name} unavailable: {str(e)[:80]}")

        # (2) Hugging Face AST (Audio Spectrogram Transformer).
        try:
            from transformers import ASTFeatureExtractor, ASTModel
            _fe = ASTFeatureExtractor.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")
            _md = ASTModel.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593").to(DEV).eval()
            def embed_ast(x, sr):
                xr = librosa.resample(x, orig_sr=sr, target_sr=16000) if sr != 16000 else x
                with torch.no_grad():
                    inp = _fe(xr, sampling_rate=16000, return_tensors="pt")
                    h = _md(**{k: v.to(DEV) for k, v in inp.items()}).last_hidden_state
                    return torch.cat([h.mean(1), h.std(1)], -1).cpu().numpy().ravel()
            EXTRACTORS["AST"] = embed_ast
            print("  AST ready")
        except Exception as e:
            print(f"  AST unavailable: {str(e)[:80]}")
    except ImportError:
        print("  torch/torchaudio not installed - baseline only")

print("\nextractors:", list(EXTRACTORS))

  WAV2VEC2_BASE ready
  HUBERT_BASE ready


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3767.75it/s]
[transformers] ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.dense.weight     | UNEXPECTED |  | 
classifier.layernorm.bias   | UNEXPECTED |  | 
classifier.dense.bias       | UNEXPECTED |  | 
classifier.layernorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  AST ready

extractors: ['Mel2023', 'WAV2VEC2_BASE', 'HUBERT_BASE', 'AST']


## Extract embeddings (cached per extractor)

In [4]:
import librosa
def embed_all(name, fn, max_sec=10.0):
    fp = cfg.CACHE / f"d2_emb_{name}.npz"
    if fp.exists():
        return np.load(fp)["X"]
    Xs = []
    for p in tqdm(D2W.path, desc=name):
        x, sr = librosa.load(p, sr=None, mono=True, duration=max_sec)
        Xs.append(fn(x, sr))
    X = np.vstack(Xs)
    np.savez_compressed(fp, X=X)
    return X

EMB = {name: embed_all(name, fn) for name, fn in EXTRACTORS.items()}
for k, v in EMB.items():
    print(f"  {k:14s} {v.shape}")

AST: 100%|██████████| 140/140 [02:11<00:00,  1.07it/s]

  Mel2023        (140, 199)
  WAV2VEC2_BASE  (140, 1536)
  HUBERT_BASE    (140, 1536)
  AST            (140, 1536)


## E7 - naive vs logger-independent, per extractor

In [5]:
y2  = D2W.label.to_numpy()
lg2 = D2W.logger.fillna("NA").to_numpy()

rows = []
for name, Xe in EMB.items():
    for seed in range(cfg.N_SEEDS):
        a, b = train_test_split(np.arange(len(y2)), test_size=.3,
                                random_state=seed, stratify=y2)
        rows.append(dict(model=name, protocol="P1_naive", seed=seed,
                         **evaluation.evaluate(Xe[a], y2[a], Xe[b], y2[b], "logreg", seed)))
        try:
            a, b = next(GroupShuffleSplit(1, test_size=.35, random_state=seed)
                        .split(np.arange(len(y2)), y2, groups=lg2))
            if len(np.unique(y2[a])) > 1 and len(np.unique(y2[b])) > 1:
                rows.append(dict(model=name, protocol="P2_logger_indep", seed=seed,
                                 **evaluation.evaluate(Xe[a], y2[a], Xe[b], y2[b], "logreg", seed)))
        except Exception:
            pass

e7 = pd.DataFrame(rows)
s7 = e7.groupby(["model", "protocol"]).agg(acc=("acc", "mean"), sd=("acc", "std"),
                                           base=("base", "mean"), n=("acc", "size")).reset_index()
print(s7.round(4).to_string(index=False))
e7.to_csv(cfg.RESULTS / "E7_foundation_raw.csv", index=False)
s7.to_csv(cfg.RESULTS / "E7_foundation_summary.csv", index=False)

piv = s7.pivot(index="model", columns="protocol", values="acc")
if piv.shape[1] >= 2:
    piv["collapse"] = piv.iloc[:, 0] - piv.iloc[:, 1]
    print("\ncollapse per model (naive - logger-independent):")
    print(piv.round(4).to_string())

        model        protocol    acc     sd   base  n
          AST        P1_naive 0.8619 0.0386 0.5000 10
          AST P2_logger_indep 0.7665 0.1175 0.6078 10
  HUBERT_BASE        P1_naive 0.8500 0.0539 0.5000 10
  HUBERT_BASE P2_logger_indep 0.7885 0.1161 0.6078 10
      Mel2023        P1_naive 0.7333 0.0708 0.5000 10
      Mel2023 P2_logger_indep 0.6745 0.0665 0.6078 10
WAV2VEC2_BASE        P1_naive 0.7857 0.0514 0.5000 10
WAV2VEC2_BASE P2_logger_indep 0.7288 0.1157 0.6078 10

collapse per model (naive - logger-independent):
protocol       P1_naive  P2_logger_indep  collapse
model                                             
AST              0.8619           0.7665    0.0954
HUBERT_BASE      0.8500           0.7885    0.0615
Mel2023          0.7333           0.6745    0.0588
WAV2VEC2_BASE    0.7857           0.7288    0.0569
